In [ ]:
# ===== 環境セットアップ =====
import os, subprocess, shutil

def _is_kaggle():
    return os.path.exists("/kaggle/input")

def _is_runpod():
    return os.environ.get("RUNPOD_POD_ID") is not None

ENV = "kaggle" if _is_kaggle() else "runpod" if _is_runpod() else "local"
print(f"[env] {ENV}")

if ENV == "runpod":
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import json as _json
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as _f:
        _json.dump({"username": os.environ["KAGGLE_USERNAME"],
                    "key": os.environ["KAGGLE_KEY"]}, _f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)


In [ ]:
# ===== パス定義 =====
import glob as _glob

def _find_path(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return candidates[0]

if ENV == "kaggle":
    SCRIPT_SRC = _find_path(
                     "/kaggle/input/datasets/shotokishida/nemotron-train-scripts/train.py",
                     "/kaggle/input/nemotron-train-scripts/train.py")
    SCRIPT_DST = "/kaggle/working/train.py"
    MODEL_DIR    = _find_path(
                       "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
                       "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1")
    COT_CSV      = _find_path(
                       "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv",
                       "/kaggle/input/nemotron-sft-lora-cot-selection/train_split_with_cot.csv",
                       "/kaggle/input/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv")
    OUTPUT_DIR   = "/kaggle/working/adapter"
elif ENV == "runpod":
    SCRIPT_SRC = None
    SCRIPT_DST = "/workspace/repo/notebooks/nemotron-train/train.py"
    MODEL_DIR    = "/workspace/models/nemotron-3-nano-30b-a3b-bf16"
    COT_CSV      = "/workspace/data/konbu17-cot/train_split_with_cot.csv"
    OUTPUT_DIR   = "/workspace/output/adapter"
else:  # local
    SCRIPT_SRC = None
    SCRIPT_DST = "notebooks/nemotron-train/train.py"
    MODEL_DIR    = None
    COT_CSV      = "data/konbu17-cot/train_split_with_cot.csv"
    OUTPUT_DIR   = "output/adapter"

# Kaggle: Input から working にコピー
if SCRIPT_SRC and os.path.exists(SCRIPT_SRC):
    shutil.copy2(SCRIPT_SRC, SCRIPT_DST)
    print(f"[script] copied: {SCRIPT_SRC} -> {SCRIPT_DST}")
elif not os.path.exists(SCRIPT_DST):
    raise FileNotFoundError(f"train.py が見つかりません: {SCRIPT_DST}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"SCRIPT     = {SCRIPT_DST}  exists={os.path.exists(SCRIPT_DST)}")
print(f"MODEL_DIR  = {MODEL_DIR}  exists={os.path.exists(str(MODEL_DIR))}")
print(f"COT_CSV    = {COT_CSV}  exists={os.path.exists(str(COT_CSV))}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

In [ ]:
# ===== ライブラリインストール =====
import glob as _g, subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# trl: Datasetのwheelからオフラインインストール（RTX Pro 6000はインターネット無効）
try:
    import trl
    print(f"trl: {trl.__version__}")
except ImportError:
    _wheels = _g.glob("/kaggle/input/datasets/shotokishida/nemotron-train-scripts/trl-*.whl") or \
              _g.glob("/kaggle/input/nemotron-train-scripts/trl-*.whl")
    if _wheels:
        print(f"Installing trl from wheel (offline): {_wheels[0]}")
        _pip("--no-deps", _wheels[0])
    else:
        print("Installing trl from PyPI (internet required)...")
        _pip("trl")
    import trl
    print(f"trl installed: {trl.__version__}")

# transformers 5.0+ は nemotron-h を公式サポート → mamba-ssm 不要

In [ ]:
# ===== データ確認 =====
import pandas as pd

df = pd.read_csv(COT_CSV)
print(f"shape: {df.shape}")
print(f"columns: {df.columns.tolist()}")
print()
print("問題タイプ別件数:")
print(df['type'].value_counts())
print()
print("--- CoT サンプル ---")
print(df['generated_cot'].iloc[0][:400])


In [ ]:
# ===== 学習実行 =====
# ここのパラメータを変えて実験する

LORA_RANK    = 32     # 16: 440M params, 32: 880M params
LORA_DROPOUT = 0.0    # Tong Hui Kang 準拠: dropout なし
EPOCHS       = 1      # 1epoch で十分（過学習回避）
LR           = 2e-4   # Tong Hui Kang 準拠
BATCH_SIZE   = 1
GRAD_ACCUM   = 32     # effective batch size = BATCH_SIZE × GRAD_ACCUM = 32
MAX_SEQ      = 8192   # 長い CoT を切らないために 8192
SAVE_STEPS   = None   # N ステップごとにチェックポイント保存。None で無効
SUBSAMPLE    = None   # 動作確認時は 100 など小さい値に

subsample_flag = f"--subsample {SUBSAMPLE}" if SUBSAMPLE else ""
save_flag      = f"--save_steps {SAVE_STEPS}" if SAVE_STEPS else ""

!python "{SCRIPT_DST}" \
    --model_dir   "{MODEL_DIR}" \
    --data_csv    "{COT_CSV}" \
    --output_dir  "{OUTPUT_DIR}" \
    --lora_rank   {LORA_RANK} \
    --lora_dropout {LORA_DROPOUT} \
    --epochs      {EPOCHS} \
    --lr          {LR} \
    --batch_size  {BATCH_SIZE} \
    --grad_accum  {GRAD_ACCUM} \
    --max_seq_len {MAX_SEQ} \
    --zip_output \
    {save_flag} \
    {subsample_flag}

In [ ]:
# ===== 学習曲線の表示（学習完了後に実行） =====
import json, glob, matplotlib.pyplot as plt

# Trainer が出力したログ（JSON 行形式）を収集
log_files = glob.glob("/kaggle/working/checkpoints/trainer_state.json") + \
            glob.glob("/kaggle/working/adapter/trainer_state.json")

metrics = []
if log_files:
    with open(log_files[0]) as f:
        state = json.load(f)
    metrics = [e for e in state.get("log_history", []) if "loss" in e]
else:
    print("trainer_state.json が見つかりません。学習完了後に実行してください。")

if metrics:
    steps = [m["step"] for m in metrics]
    loss  = [m["loss"] for m in metrics]
    lr    = [m["learning_rate"] for m in metrics]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(steps, loss, color="steelblue")
    ax1.set_title("Training Loss")
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss")
    ax1.grid(True)

    ax2.plot(steps, lr, color="darkorange")
    ax2.set_title("Learning Rate")
    ax2.set_xlabel("Step")
    ax2.set_ylabel("LR")
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curve.png", dpi=120)
    plt.show()
    print(f"グラフ保存: /kaggle/working/training_curve.png")
    print(f"最終 loss: {loss[-1]:.4f}  ({len(steps)} steps)")


In [ ]:
# ===== 出力確認 =====
print("adapter files:", os.listdir(OUTPUT_DIR))

zip_path = os.path.join(os.path.dirname(OUTPUT_DIR), 'submission.zip')
if os.path.exists(zip_path):
    import zipfile
    with zipfile.ZipFile(zip_path) as zf:
        print("submission.zip contents:", zf.namelist())
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f"submission.zip size: {size_mb:.1f} MB")
